# MSS historical data quality
Reproduce the frozen-file audit without modifying raw files or executing any strategy.
Run from the repository or this reports directory. Chart contract: one ranked bar chart across eight symbols, zero recorded spread share, denominator excluding warmup; no causal or net-cost interpretation.
Report structure follows technical summary, scope, findings, methods, caveats and next actions. Historical closure and contract-cost evidence remains unresolved.

In [ ]:
from pathlib import Path
import sys
root = Path.cwd()
if not (root / 'src').exists():
    root = root.parent
sys.path.insert(0, str(root / 'src'))
from mss.analysis.historical_dataset_quality import audit_dataset
audit = audit_dataset(root)
assert audit['all_integrity_checks_pass']
audit['blockers']

In [ ]:
import sqlite3, json
sql = "SELECT json_extract(value, '$.symbol') AS symbol, json_extract(value, '$.performance_rows') AS rows, COALESCE(json_extract(value, '$.zero_counts_performance_only.spread'), 0) AS zeroSpreadRows, 1.0 * COALESCE(json_extract(value, '$.zero_counts_performance_only.spread'), 0) / json_extract(value, '$.performance_rows') AS zeroSpreadRate, json_extract(value, '$.internal_gap_count') AS gaps, json_extract(value, '$.internal_missing_slots') AS missingSlots, json_extract(value, '$.internal_weekend_slots') AS weekendSlots FROM json_each(:audit, '$.symbols') ORDER BY zeroSpreadRate DESC"
with sqlite3.connect(':memory:') as conn:
    conn.row_factory = sqlite3.Row
    profile = [dict(r) for r in conn.execute(sql, {'audit':json.dumps(audit)})]
profile